# Solving Poisson's equation on analytic and spline-mapped domains

This notebook shows the two ways PSYDAC can describe the geometry of a mapped domain, and solves the same PDE on both:

- an **analytic mapping** — the map $F:\widehat{\Omega}\to\Omega$ is given by closed-form expressions (here `PolarMapping`);
- a **spline mapping** — $F$ is represented by B-splines, e.g. because the geometry comes from CAD or was built by interpolation.

The model is identical in both cases. What changes is the object that carries the geometry, and therefore *how PSYDAC assembles the equations*: from the analytic Jacobian, or by evaluating the spline on the quadrature grid.

The domain is two quarter-annuli glued along an edge, and we solve Poisson's equation with a manufactured solution, using a symmetric interior-penalty (SIPG) coupling across the interface.

## Step 1: An analytic mapping

A `PolarMapping` sends logical coordinates $(\eta_1,\eta_2)$ to
$$
x = c_1 + \big((1-\eta_1)\, r_\text{min} + \eta_1\, r_\text{max}\big)\cos\eta_2, \qquad
y = c_2 + \big((1-\eta_1)\, r_\text{min} + \eta_1\, r_\text{max}\big)\sin\eta_2 ,
$$
so $\eta_1$ interpolates the radius from $r_\text{min}$ to $r_\text{max}$ and $\eta_2$ is the angle.

It is an `AnalyticMapping`: it has a symbolic identity (a name, it can be applied to a domain), **and** it can be evaluated at points, because it carries its own expressions.

In [ ]:
import numpy as np

from sympde.topology import Square, PolarMapping

rmin, rmax = 0.3, 1.0

# Two logical patches, and one analytic mapping for each
A_1 = Square('A_1', bounds1=(0., 1.), bounds2=(0.,    0.5 * np.pi))
A_2 = Square('A_2', bounds1=(0., 1.), bounds2=(np.pi, 1.5 * np.pi))

F_1 = PolarMapping('F_1', dim=2, c1=0.,          c2=0., rmin=rmin, rmax=rmax)
F_2 = PolarMapping('F_2', dim=2, c1=rmin + rmax, c2=0., rmin=rmax, rmax=rmin)

print('F_1(0.5, 0.7)          =', F_1(0.5, 0.7))
print('F_1.is_analytical      =', F_1.is_analytical)
print('get_callable_mapping() is F_1:', F_1.get_callable_mapping() is F_1)

An analytic mapping evaluates arrays too, so it can be applied to a whole grid at once:

In [ ]:
eta1 = np.linspace(0, 1, 41)
eta2 = np.linspace(0, 0.5 * np.pi, 41)
E1, E2 = np.meshgrid(eta1, eta2, indexing='ij')

x_a, y_a = F_1(E1, E2)
print('grid shape:', x_a.shape)

## Step 2: A spline approximation of the same geometry

`SplineCallableMapping.from_mapping` interpolates any point-evaluable mapping onto a B-spline space. Here we approximate each `PolarMapping` with degree-2 splines on an $8\times 8$ grid.

The result is a **`SplineCallableMapping`**: it can be evaluated at points, but it has **no symbolic identity** — no name, and it cannot be applied to a domain or used in a weak form.

In [ ]:
from psydac.mapping.discrete import SplineCallableMapping

def spline_approx(F, logical_domain):
    # The symbolic mapping F can be passed directly: from_mapping unwraps it.
    return SplineCallableMapping.from_mapping(
        F, ncells=(8, 8), degree=(2, 2),
        bounds=zip(logical_domain.min_coords, logical_domain.max_coords))

F_1h = spline_approx(F_1, A_1)
F_2h = spline_approx(F_2, A_2)

print('F_1h(0.5, 0.7) =', F_1h(0.5, 0.7))
print('F_1h.name      =', F_1h.name, '  <- no symbolic identity')

How close is the spline to the analytic geometry? Evaluate both on the same grid:

In [ ]:
x_s, y_s = F_1h(E1, E2)
geometry_error = np.max(np.hypot(x_a - x_s, y_a - y_s))
print(f'max geometric error over the 41 x 41 grid: {geometry_error:.2e}')

## Step 3: Giving the spline a symbolic identity

To use the spline in a domain and in a weak form, wrap it in a **`DiscreteMapping`** with `to_defined_mapping(name)`. This creates a new symbolic mapping whose point evaluation is delegated to the spline. Nothing is modified in place.

The key property is `is_analytical`: it is `False`, which tells PSYDAC to assemble by **evaluating the spline on the quadrature grid** rather than using a symbolic Jacobian.

In [ ]:
M_1 = F_1h.to_defined_mapping('F_1')
M_2 = F_2h.to_defined_mapping('F_2')

print('type(M_1)              =', type(M_1).__name__)
print('M_1.is_analytical      =', M_1.is_analytical)
print('M_1.get_callable_mapping() is F_1h:', M_1.get_callable_mapping() is F_1h)

## Step 4: Building the two multipatch domains

Both domains are built the same way: apply each mapping to its logical patch, then glue the two patches with `Domain.join`. The connectivity says that edge $(\text{axis }1, \text{ext }-1)$ of patch 0 is glued to edge $(\text{axis }1, \text{ext }-1)$ of patch 1.

In [ ]:
from sympde.topology import Domain
from sympde.utilities.utils import plot_domain

connectivity = [((0, 1, -1), (1, 1, -1), 1)]

Omega_analytic = Domain.join([F_1(A_1), F_2(A_2)], connectivity, 'domain')
Omega_spline   = Domain.join([M_1(A_1), M_2(A_2)], connectivity, 'domain')

plot_domain(Omega_analytic, draw=False, isolines=True)

In [ ]:
plot_domain(Omega_spline, draw=False, isolines=True)

## Step 5: The abstract model

We solve
$$
-\Delta u = f \quad\text{in } \Omega, \qquad u = g \quad\text{on } \partial\Omega,
$$
with the manufactured solution $u = x^2 + y^2$, so that $f = -4$ and $g = u|_{\partial\Omega}$.

The Dirichlet condition is imposed weakly (Nitsche), and continuity across the interface by a symmetric interior-penalty (SIPG) term with penalty $\kappa$. The model depends only on the **symbolic** domain, so we write it once and reuse it for both geometries.

In [ ]:
from sympde.calculus import grad, dot, minus, plus
from sympde.topology import ScalarFunctionSpace, elements_of, NormalVector
from sympde.expr     import BilinearForm, LinearForm, integral, Norm, find

def poisson_model(Omega, kappa=1e3):
    x, y     = Omega.coordinates
    solution = x**2 + y**2
    f        = -4

    V    = ScalarFunctionSpace('V', Omega, kind=None)
    u, v = elements_of(V, names='u, v')
    nn   = NormalVector('nn')
    I    = Omega.interfaces
    bnd  = Omega.boundary

    # SIPG coupling across the interface
    expr_I = (- 0.5*dot(grad(plus(u)), nn)*minus(v)  + 0.5*dot(grad(minus(v)), nn)*plus(u)  - kappa*plus(u)*minus(v)
              + 0.5*dot(grad(minus(u)), nn)*plus(v)  - 0.5*dot(grad(plus(v)), nn)*minus(u)  - kappa*plus(v)*minus(u)
              - 0.5*dot(grad(minus(v)), nn)*minus(u) - 0.5*dot(grad(minus(u)), nn)*minus(v) + kappa*minus(u)*minus(v)
              + 0.5*dot(grad(plus(v)), nn)*plus(u)   + 0.5*dot(grad(plus(u)), nn)*plus(v)   + kappa*plus(u)*plus(v))
    # Nitsche boundary term
    expr_b = -dot(grad(u), nn)*v - dot(grad(v), nn)*u + kappa*u*v

    a = BilinearForm((u, v), integral(Omega, dot(grad(u), grad(v)))
                             + integral(I, expr_I) + integral(bnd, expr_b))
    l = LinearForm(v, integral(Omega, f*v)
                      + integral(bnd, -dot(grad(v), nn)*solution + kappa*solution*v))

    equation = find(u, forall=v, lhs=a(u, v), rhs=l(v))
    l2norm   = Norm(u - solution, Omega, kind='l2')
    return V, equation, l2norm

## Step 6: Discretizing — where the two cases differ

This is the one place the two geometries are handled differently:

- The **analytic** domain carries no discretization of its own, so we choose one: `discretize(Omega, ncells=..., periodic=...)`, and a spline degree for the solution space.
- The **spline** domain *already carries* a discrete geometry. `discretize(Omega)` with no `filename` and no `ncells` builds the discrete geometry from the splines themselves, and the solution space inherits the splines' own grid and degree.

In [ ]:
from psydac.api.discretization import discretize
from psydac.api.settings       import PSYDAC_BACKENDS

# The pure-Python backend needs no compilation. For speed, use a pyccel backend
# instead, e.g. PSYDAC_BACKENDS['pyccel-gcc'].
backend = PSYDAC_BACKENDS['python']

def discretize_problem(Omega, V, equation, l2norm, *, spline):
    if spline:
        Omega_h = discretize(Omega)                        # geometry from the splines
        Vh      = discretize(V, Omega_h)                   # grid and degree inherited
    else:
        Omega_h = discretize(Omega, ncells=[10, 10], periodic=[False, False])
        Vh      = discretize(V, Omega_h, degree=[2, 2])
    equation_h = discretize(equation, Omega_h, [Vh, Vh], backend=backend)
    l2norm_h   = discretize(l2norm,   Omega_h, Vh,       backend=backend)
    return Vh, equation_h, l2norm_h

## Step 7: Solving and computing the errors

In [ ]:
import time

results = {}
for label, Omega, spline in [('analytic', Omega_analytic, False),
                             ('spline',   Omega_spline,   True)]:
    V, equation, l2norm   = poisson_model(Omega)
    Vh, equation_h, l2norm_h = discretize_problem(Omega, V, equation, l2norm, spline=spline)

    t0 = time.time()
    uh = equation_h.solve()
    l2_error = float(l2norm_h.assemble(u=uh))

    p = Vh.spaces[0]
    results[label] = dict(Omega=Omega, uh=uh, l2_error=l2_error)
    print(f'{label:9s}: L2 error = {l2_error:.3e}   '
          f'[{list(p.ncells)} cells/patch, degree {list(p.degree)}, {Vh.nbasis} dofs, '
          f'{time.time() - t0:.1f} s]')

l2_error_analytic = results['analytic']['l2_error']
l2_error_spline   = results['spline']['l2_error']

**Reading these numbers.** Do not read this as "the spline geometry is more accurate". The two runs do not use the same discrete space: the analytic run was given $10\times 10$ cells per patch, while the spline run inherits its geometry's coarser $8\times 8$ grid. The errors therefore mix two different effects — the approximation of the *solution* and, in the spline case, the approximation of the *geometry* — and are not a like-for-like comparison.

What the example does show is that the same abstract model runs unchanged on both geometries, and that both converge to the manufactured solution.

## Step 8: Plotting the solutions

In [ ]:
from psydac.fem.plotting_utilities import plot_field_2d

for label, r in results.items():
    # filename=None: show the plot inline without also writing an image file.
    # cmap='viridis': a sequential colormap reads better for a scalar field than
    # the default cyclic 'hsv', whose two ends have nearly the same hue.
    plot_field_2d(fem_field=r['uh'], domain=r['Omega'], filename=None, cmap='viridis',
                  title=f'Poisson solution, {label} mapping', hide_plot=False)

### Testing the notebook

In [ ]:
import ipytest
ipytest.autoconfig(raise_on_error=True)

In [ ]:
%%ipytest

def test_analytic_l2error():
    assert l2_error_analytic < 2e-4

def test_spline_l2error():
    assert l2_error_spline < 1e-3

def test_spline_geometry_error():
    assert geometry_error < 5e-4